# Osobní rozpočet: dojdou mi peníze?

*Nad rámec skripta — aplikace.*

---

Pojišťovna z cvičení 2 a 9 vybírá pojistné a vyplácí škody. Vy dostáváte výplatu a platíte nájem, jídlo, občas restauraci a jednou za čas vám praskne pračka.

**Je to tatáž úloha.** Příjem = pojistné, výdaje = škody, rezerva na účtu = kapitál, „dojdou peníze" = úpadek. Celý aparát z pojišťovny se přenese beze změny — jen je to tentokrát váš rozpočet.

Přidáme ale jednu otázku, kterou pojišťovna neměla: **spořím si na auto za 100 000. Dám to dřív, než se dostanu do mínusu?** To jsou dvě hranice místo jedné, a ta úloha má krásný přesný výsledek — je to klasická **ruina hráče**.

V sešitu uvidíte:

- model rozpočtu: pevné platby, denní nákupy, restaurace jako složený proces (cvičení 5), poruchy jako vzácné velké výdaje
- **jak velkou rezervu** potřebuju, aby mi do dvou let nedošly peníze
- **dvě hranice**: cíl versus mínus, a přesné řešení ruiny hráče jako kontrola
- **co riziko vlastně žene** — a je to něco jiného, než by člověk čekal

In [ ]:
import math
import random

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def mezery(x, spec=",.0f"):
    return format(x, spec).replace(",", " ")


def poisson(lam):
    """Počet událostí: součin rovnoměrných čísel pod e^-λ (cvičení 3)."""
    L, p, k = math.exp(-lam), 1.0, 0
    while True:
        p *= random.random()
        if p < L:
            return k
        k += 1


# --- parametry rozpočtu (měsíčně, v korunách) ---
PRIJEM = 38_000            # čistá mzda
NAJEM = 16_000             # nájem včetně energií
OSTATNI_PEVNE = 4_000      # doprava, telefon, předplatné

JIDLO_DEN, JIDLO_SD = 230.0, 90.0        # běžný nákup, kolísá ze dne na den
RESTAURACE_MESICNE = 7                   # kolikrát měsíčně jdu jíst ven
RESTAURACE_OD, RESTAURACE_DO = 250, 900

PORUCHY_ROCNE = 2.5        # pračka, lednička, oprava v bytě…
PORUCHA_ZAKLAD = 3_000     # nic se nespraví pod tři tisíce
PORUCHA_STREDNI = 9_000    # a navíc exponenciální chvost


def vydaje_za_mesic(bez_nahody=()):
    """Výdaje za jeden měsíc.

    bez_nahody: názvy složek, které se mají nahradit jejich střední hodnotou
    (použijeme v oddílu 4, abychom zjistili, která nejistota vlastně rozhoduje).
    """
    v = NAJEM + OSTATNI_PEVNE

    if "jidlo" in bez_nahody:
        v += 30 * JIDLO_DEN
    else:
        v += sum(max(0.0, random.gauss(JIDLO_DEN, JIDLO_SD)) for _ in range(30))

    if "restaurace" in bez_nahody:
        v += RESTAURACE_MESICNE * (RESTAURACE_OD + RESTAURACE_DO) / 2
    else:
        for _ in range(poisson(RESTAURACE_MESICNE)):
            v += random.uniform(RESTAURACE_OD, RESTAURACE_DO)

    if "poruchy" in bez_nahody:
        v += PORUCHY_ROCNE / 12 * (PORUCHA_ZAKLAD + PORUCHA_STREDNI)
    else:
        for _ in range(poisson(PORUCHY_ROCNE / 12)):
            v += PORUCHA_ZAKLAD + random.expovariate(1 / PORUCHA_STREDNI)

    return v


# přesná střední hodnota měsíčních výdajů
E_VYDAJE = (NAJEM + OSTATNI_PEVNE + 30 * JIDLO_DEN
            + RESTAURACE_MESICNE * (RESTAURACE_OD + RESTAURACE_DO) / 2
            + PORUCHY_ROCNE / 12 * (PORUCHA_ZAKLAD + PORUCHA_STREDNI))
print(f"očekávané výdaje za měsíc: {mezery(E_VYDAJE)} Kč")
print(f"příjem {mezery(PRIJEM)} Kč  →  v průměru odložím {mezery(PRIJEM - E_VYDAJE)} Kč měsíčně")

## 1. Jak vypadá měsíc

Model má čtyři druhy výdajů a každý se chová jinak:

| složka | model | odkud |
|---|---|---|
| nájem, telefon, doprava | pevná částka | — |
| běžné nákupy | 30 denních částek, každá kolem 230 Kč | součet mnoha malých ⟹ CLV (cvičení 0) |
| restaurace | náhodný **počet** × náhodná **částka** | složený proces (cvičení 5) |
| poruchy | vzácné, ale velké | Poisson (cvičení 3) + exponenciální chvost |

Jídlo kolísá ze dne na den, ale je jich třicet, takže se to zprůměruje. Poruchy jsou opačný případ: většinu měsíců nula, občas rána. Podívejme se, jak vypadá rozdělení měsíčních výdajů jako celku.

In [ ]:
n = 100_000
vzorky = [vydaje_za_mesic() for _ in range(n)]
m = sum(vzorky) / n
sd = math.sqrt(sum((x - m) ** 2 for x in vzorky) / (n - 1))
print(f"měsíční výdaje: průměr {mezery(m)} Kč (přesně {mezery(E_VYDAJE)}), "
      f"směrodatná odchylka {mezery(sd)} Kč")
assert abs(m - E_VYDAJE) < 5 * sd / math.sqrt(n)
print(f"nad příjem {mezery(PRIJEM)} Kč se dostane {sum(1 for x in vzorky if x > PRIJEM) / n:.1%} měsíců")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))

ax = axes[0]
w = 60_000 / 90
ax.hist(vzorky, bins=90, range=(0, 60_000), weights=[1 / (n * w)] * n, alpha=0.6)
ax.axvline(PRIJEM, color="tab:red", ls="--", label=f"příjem {mezery(PRIJEM)}")
ax.axvline(E_VYDAJE, color="k", ls=":", label=f"průměrné výdaje {mezery(E_VYDAJE)}")
ax.set_xlabel("výdaje za měsíc [Kč]")
ax.set_ylabel("hustota")
ax.set_title("Většinu měsíců zbyde, občas ne", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
random.seed(4)
for _ in range(12):
    stav, cesta = 20_000, [20_000]
    for _ in range(24):
        stav += PRIJEM - vydaje_za_mesic()
        cesta.append(stav)
    ax.plot(cesta, lw=1, alpha=0.8)
ax.axhline(0, color="k", lw=1)
ax.axhline(100_000, color="tab:green", ls="--", lw=1, label="cíl 100 000 (auto)")
ax.set_xlabel("měsíc")
ax.set_ylabel("stav účtu [Kč]")
ax.set_title("Dvanáct variant dvou let, rezerva 20 000", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Pravý graf je celá úloha v jednom obrázku. Většina drah stoupá — v průměru se přece odkládá skoro pět tisíc měsíčně — ale některé spadnou pod nulu dřív, než se vyšplhají k cíli. A tvar těch poklesů prozradí původce: nejsou to pozvolné sešupy, ale **schody**. Poruchy.

## 2. Jak velkou rezervu potřebuju?

Stejná otázka jako „kolik kapitálu potřebuje pojišťovna" ze cvičení 9, jen menší čísla.

In [ ]:
def dojdou_penize(rezerva, mesicu=24, bez_nahody=(), M_=20_000):
    """Podíl běhů, ve kterých stav účtu někdy klesne pod nulu."""
    cnt = 0
    for _ in range(M_):
        stav = rezerva
        for _ in range(mesicu):
            stav += PRIJEM - vydaje_za_mesic(bez_nahody)
            if stav < 0:
                cnt += 1
                break
    return cnt / M_


print(f"{'rezerva':>10s} {'v měsících výdajů':>18s} {'P(dojdou peníze do 2 let)':>27s}")
rezervy = (0, 10_000, 20_000, 40_000, 60_000, 100_000)
p_dojde = []
for r in rezervy:
    p = dojdou_penize(r)
    p_dojde.append(p)
    print(f"{mezery(r):>10s} {r / E_VYDAJE:>17.2f} {p:>27.4f}")

cil = 0.01
vhodna = next(r for r, p in zip(rezervy, p_dojde) if p < cil)
print(f"\nPod {cil:.0%} riziko se dostaneme s rezervou kolem {mezery(vhodna)} Kč,")
print(f"což jsou zhruba {vhodna / E_VYDAJE:.1f} měsíce výdajů.")
print(f"\nPoradci obvykle radí rezervu na 3 až 6 měsíců, tedy podstatně víc. Není to spor:")
print(f"náš model totiž předpokládá, že příjem chodí vždy. Jakmile připustíme ztrátu")
print(f"zaměstnání, rezerva musí pokrýt celé výdaje, ne jen jejich kolísání — a požadavek")
print(f"vyskočí na několikanásobek. Je to cvičení 2 na konci sešitu.")

## 3. Dvě hranice: cíl, nebo mínus?

Teď otázka, kterou pojišťovna nemá. Spořím na auto za 100 000 a mám rezervu $u_0$. Zajímá mě:

> Dosáhnu cíle dřív, než spadnu do mínusu?

To jsou **dvě absorbující hranice** — nahoře cíl $G$, dole nula — a v jednoduché podobě je to klasická **ruina hráče** s přesným řešením. Hráč začíná s $u_0$ a v každém kroku vyhraje korunu s pravděpodobností $p$, jinak ji prohraje. Pravděpodobnost, že dojde k $G$ dřív než k nule, je

$$P = \frac{1 - r^{u_0}}{1 - r^{G}}, \qquad r = \frac{1-p}{p} \qquad (\text{a } P = u_0/G \text{ pro } p = \tfrac12).$$

Na tenhle vzorec si nejdřív ověříme simulátor — je to jediná část úlohy, kde známe přesnou odpověď.

In [ ]:
def ruina_hrace_sim(u0, G, p, M_=100_000):
    """Simulace: ±1 za krok, absorpce v 0 a v G."""
    uspech = 0
    for _ in range(M_):
        x = u0
        while 0 < x < G:
            x += 1 if random.random() < p else -1
        uspech += x >= G
    return uspech / M_


def ruina_hrace_presne(u0, G, p):
    if abs(p - 0.5) < 1e-12:
        return u0 / G
    r = (1 - p) / p
    return (1 - r**u0) / (1 - r**G)


print(f"{'u_0':>5s} {'G':>5s} {'p':>6s} {'simulace':>11s} {'přesně':>10s}")
for u0, G, p in ((5, 20, 0.50), (5, 20, 0.52), (10, 30, 0.48), (3, 10, 0.55)):
    sim = ruina_hrace_sim(u0, G, p, 60_000)
    pres = ruina_hrace_presne(u0, G, p)
    print(f"{u0:>5d} {G:>5d} {p:>6.2f} {sim:>11.5f} {pres:>10.5f}")
    assert abs(sim - pres) < 0.01

print("\nSimulátor dvou hranic je ověřený. Teď totéž s opravdovým rozpočtem,")
print("kde krok není ±1, ale měsíční přebytek s celým svým rozdělením.")

In [ ]:
def dosahnu_cile(rezerva, cil_castka=100_000, mesicu=36, bez_nahody=(), M_=20_000):
    """Vrátí (P(dosáhnu cíle dřív než mínusu), P(mínus), P(ani jedno do konce))."""
    uspech = krach = 0
    for _ in range(M_):
        stav = rezerva
        for _ in range(mesicu):
            stav += PRIJEM - vydaje_za_mesic(bez_nahody)
            if stav < 0:
                krach += 1
                break
            if stav >= cil_castka:
                uspech += 1
                break
    return uspech / M_, krach / M_, 1 - (uspech + krach) / M_


print(f"cíl 100 000 Kč, horizont 3 roky\n")
print(f"{'rezerva':>10s} {'dosáhnu cíle':>14s} {'spadnu do mínusu':>18s} {'ani jedno':>11s}")
rezervy2 = (0, 10_000, 20_000, 40_000, 60_000)
vysl = {}
for r in rezervy2:
    u, k, n_ = dosahnu_cile(r)
    vysl[r] = (u, k, n_)
    print(f"{mezery(r):>10s} {u:>14.3f} {k:>18.3f} {n_:>11.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax = axes[0]
ax.plot(rezervy2, [vysl[r][0] for r in rezervy2], "o-", color="tab:green", label="dosáhnu cíle")
ax.plot(rezervy2, [vysl[r][1] for r in rezervy2], "s-", color="tab:red", label="spadnu do mínusu")
ax.plot(rezervy2, [vysl[r][2] for r in rezervy2], "^-", color="tab:gray", label="ani jedno do 3 let")
ax.set_xlabel("počáteční rezerva [Kč]")
ax.set_ylabel("pravděpodobnost")
ax.set_title("Vyšší rezerva = víc šancí dojít k cíli", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
random.seed(11)
for _ in range(80):
    stav, cesta = 20_000, [20_000]
    for _ in range(36):
        stav += PRIJEM - vydaje_za_mesic()
        cesta.append(stav)
        if stav < 0 or stav >= 100_000:
            break
    barva = "tab:green" if cesta[-1] >= 100_000 else ("tab:red" if cesta[-1] < 0 else "0.75")
    ax.plot(cesta, lw=0.8, alpha=0.7, color=barva)
ax.axhline(0, color="k", lw=1)
ax.axhline(100_000, color="tab:green", ls="--", lw=1)
ax.set_xlabel("měsíc")
ax.set_ylabel("stav účtu [Kč]")
ax.set_title("80 drah s rezervou 20 000 (zelená = cíl, červená = mínus)", fontsize=10)
plt.tight_layout()
plt.show()

## 4. Co riziko vlastně žene

Praktická otázka: **na čem se vyplatí šetřit?** Chodit míň do restaurace? Nakupovat levněji? Nebo je to něco jiného?

Odpověď dá jednoduchý pokus. Postupně nahradíme každou náhodnou složku její **střední hodnotou** — tedy odebereme jí nejistotu, ale ne náklad — a podíváme se, o kolik klesne riziko. Co riziko nezmění, to na něm nestojí.

In [ ]:
print(f"rezerva 20 000, horizont 2 roky\n")
zaklad_p = dojdou_penize(20_000)
print(f"{'co zbavíme nejistoty':>28s} {'P(dojdou peníze)':>18s} {'pokles':>10s}")
print(f"{'nic (plný model)':>28s} {zaklad_p:>18.4f} {'—':>10s}")
for slozka, popis in (("jidlo", "běžné nákupy"),
                      ("restaurace", "restaurace"),
                      ("poruchy", "poruchy a opravy")):
    p = dojdou_penize(20_000, bez_nahody=(slozka,))
    print(f"{popis:>28s} {p:>18.4f} {1 - p / zaklad_p:>9.0%}")

p_vse = dojdou_penize(20_000, bez_nahody=("jidlo", "restaurace", "poruchy"))
print(f"{'všeho (deterministický)':>28s} {p_vse:>18.4f} {1 - p_vse / zaklad_p:>9.0%}")

e_poruchy = mezery(PORUCHY_ROCNE / 12 * (PORUCHA_ZAKLAD + PORUCHA_STREDNI))
e_restaurace = mezery(RESTAURACE_MESICNE * (RESTAURACE_OD + RESTAURACE_DO) / 2)
print(f"\nZbavit nejistoty jídlo ani restauraci skoro nepomůže. Poruchy ano — a to přestože")
print(f"stojí v průměru jen {e_poruchy} Kč měsíčně, zatímco restaurace {e_restaurace} Kč.")
print(f"\nO riziku nerozhoduje, kolik co stojí v průměru, ale jak moc to KOLÍSÁ.")

Tohle je stejný závěr jako u shluků v pojišťovně, jen převedený do praktické rady: **o riziku nerozhoduje průměrný náklad, ale jeho rozptyl.**

Restaurace stojí měsíčně skoro dvakrát tolik co poruchy, a přesto na riziku nestojí — protože je to spolehlivá, předvídatelná částka. Poruchy jsou levnější a zároveň nebezpečnější, protože přijdou najednou a celé.

Praktický důsledek: pokud chcete snížit riziko, že vám dojdou peníze, přestat chodit do restaurace je **neúčinné**. Účinné je buď mít rezervu, nebo to riziko přesunout někam jinam — což je přesně to, čemu se říká pojištění. Tím se kruh uzavírá: pojišťovna z cvičení 2 existuje proto, aby za poplatek převzala právě tenhle rozptyl.

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Rozpočet = pojišťovna | příjem ↔ pojistné, výdaje ↔ škody, rezerva ↔ kapitál, mínus ↔ úpadek |
| Složený proces | restaurace: náhodný počet × náhodná částka (cvičení 5) |
| Vzácné velké výdaje | poruchy: Poisson + exponenciální chvost (cvičení 3 a 4) |
| Dvě hranice | cíl vs. mínus = **ruina hráče**, $P = (1 - r^{u_0})/(1 - r^{G})$ |
| Velikost rezervy | ~2 měsíce výdajů na 1% riziko — ale jen při jistém příjmu |
| Co žene riziko | **rozptyl, ne průměr**: poruchy jsou levnější než restaurace a přesto nebezpečnější |
| K čemu je pojištění | přesun rozptylu na někoho, kdo ho unese (a kdo řeší cvičení 9) |

### Zkuste si

1. **Nepravidelný příjem.** Místo pevné mzdy ať příjem kolísá (brigády, faktury). O kolik vzroste potřebná rezerva při stejné střední hodnotě?
2. **Ztráta zaměstnání.** Přidejte riziko, že s pravděpodobností 2 % za měsíc přijdete o práci a další 4 měsíce máte jen poloviční příjem. Vyjde teď doporučovaná rezerva na 3–6 měsíců výdajů?
3. **Inflace.** Ať výdaje rostou o 0.3 % měsíčně, příjem jednou ročně skokem. Jak se změní doba do cíle?
4. Ověřte vzorec ruiny hráče pro nesymetrickou procházku s krokem $\pm a$ místo $\pm 1$. Platí pořád?
5. Spočítejte **střední dobu** do dosažení cíle (ne jen pravděpodobnost) a porovnejte s naivním odhadem $G/(\text{měsíční přebytek})$. Proč je skutečnost delší? (Nápověda: Jensenova nerovnost.)
6. Zaveďte spoření s úrokem: přebytek se zhodnocuje o 0.4 % měsíčně. Jde o **multiplikativní** proces — liší se tím úloha podstatně, nebo ne?
7. Nahraďte poruchy pojištěním: platte 1 200 Kč měsíčně navíc a poruchy ať jsou zdarma. Vyplatí se to z hlediska rizika? A z hlediska průměru?